In [6]:
import pandas as pd
import numpy as np

# 10.1001_jamanetworkopen.2019.20511

In [7]:

mturk_file = "10.1001_jamanetworkopen.2019.20511/ptbias_lucid.csv"
lucid_file = "10.1001_jamanetworkopen.2019.20511/ptbias_mturk.csv"
df_mturk = pd.read_csv(mturk_file)
df_lucid = pd.read_csv(lucid_file)

# Add study identifier
df_mturk['study_source'] = 'MTurk'
df_lucid['study_source'] = 'Lucid'

# Column mapping from R variable names to JSON names
column_mapping = {
    # Treatment variable
    'Z': 'group',
    
    # Demographics (ordinal/continuous in JSON)
    'dem_age': 'Age',
    'health_mental': 'Mental health',
    'health_overall': 'Overall health',
    'dem_trustdoc': 'Trust in physicians',
    
    # Primary outcomes (ordinal in JSON)
    'y_confdoc': 'Patient confidence',
    'y_satis': 'Patient satisfaction', 
    'y_rcmnd': 'Likelihood to recommend',
    'y_askmor': 'Requests more tests',
    'y_whichd': 'Believes symptom checker (physician)',
    
    # Secondary outcomes
    'y_warm_index': 'Perceived warmth',
    'y_comp_index': 'Perceived competence',
    
    # Composite outcome (continuous in JSON)
    'primary_dvs_index': 'Composite index',
    
    # Demographics (binary in JSON)
    'dem_female': 'Female',
    'dem_college': 'College educated',
    'insure_unpaid': 'Unpaid medical bills',
    
    # Race/ethnicity (categorical/binary)
    'dem_race4': 'Race/ethnicity',
    
    # Insurance (categorical/binary) 
    'insure_type': 'Insurance',
    'insure_cover': 'insurance_coverage',
    
    # Health variables
    'health_pastvisit': '\u001e1 Emergency department visit in past 6 mo',
    
    # Study-specific variables (Lucid only) 
    'y_error_complain': 'Willingness to complain',
    'y_error_sue': 'Willingness to sue',
    
    # Keep some original names for reference
    'R': 'analysis_sample',
    'S': 'original_study_id'
}

# Find common columns between datasets
common_cols = set(df_mturk.columns).intersection(set(df_lucid.columns))
mturk_only = set(df_mturk.columns) - common_cols
lucid_only = set(df_lucid.columns) - common_cols


# Combine datasets (outer join to keep all columns)
print("Combining datasets...")
df_combined = pd.concat([df_mturk, df_lucid], ignore_index=True, sort=False)

print("Renaming columns...")
df_combined = df_combined.rename(columns=column_mapping)

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# Define the 5 preregistered primary outcome measures
primary_outcomes = [
    'Patient confidence',
    'Patient satisfaction', 
    'Likelihood to recommend',
    'Believes symptom checker (physician)',
    'Requests more tests'
]

# Create a mask to only compute PCA where all 5 measures are present
valid_rows = df_combined[primary_outcomes].notna().all(axis=1)

if valid_rows.any():
    # Compute the first principal component
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(df_combined.loc[valid_rows, primary_outcomes])
    
    # Scale the result to a 0-100 range
    scaler = MinMaxScaler(feature_range=(0, 100))
    scaled_index = scaler.fit_transform(pc1).flatten()
    
    # PCA direction is arbitrary. Ensure higher values represent better outcomes
    # Check correlation with 'Patient satisfaction' to determine if we need to flip the scale
    corr = np.corrcoef(df_combined.loc[valid_rows, 'Patient satisfaction'], scaled_index)[0, 1]
    if corr < 0:
        scaled_index = 100 - scaled_index
    df_combined.loc[valid_rows, 'composite index'] = scaled_index
    


# Create income below median indicator (assuming you have income data)
if 'dem_income' in df_combined.columns:
    # You'll need to define what "below median" means for your data
    median_income_category = df_combined['dem_income'].median()
    df_combined['Household income below median level'] = (df_combined['dem_income'] <= median_income_category).astype(int)    
    
# Create treatment group mapping
if 'group' in df_combined.columns:
    treatment_mapping = {
        'WM': "White Male", 
        'WF': "White Female",
        'BM': "Black Male",
        'BF': "Black Female",
    }
    df_combined['group'] = df_combined['group'].map(treatment_mapping)


# safe as 10.1001_jamanetworkopen.2019.20511/10.1001_jamanetworkopen.2019.20511.csv
output_file = "10.1001_jamanetworkopen.2019.20511/10.1001_jamanetworkopen.2019.20511.csv"
df_combined.to_csv(output_file, index=False)


columns_to_drop = [
    # MTurk undescribed variables
    'pcp_binary',    # Binary indicator for having a primary care physician
    'pcp_sex',       # Gender of participant's primary care physician  
    'y_warm',        # Alternative warmth composite (different from y_warm_index)
    'y_comp',        # Alternative competence composite (different from y_comp_index)
    
    # Lucid undescribed variables
    'health_trustdoc',  # Alternative trust in doctors measure
    'y_error_level',    # Error severity level measure
    'y_error_punitive'  # Punitive response measure for medical errors
]

existing_columns_to_drop = [col for col in columns_to_drop if col in df_combined.columns]
df_combined = df_combined.drop(columns=existing_columns_to_drop)

output_file = "10.1001_jamanetworkopen.2019.20511/10.1001_jamanetworkopen.2019.20511_clean.csv"
df_combined.to_csv(output_file, index=False)

Combining datasets...
Renaming columns...


/Users/jonasgottal/Documents/Research/Text2Tabular-Reconstructing-Research-Data-from-Scientific-Publications/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jonasgottal/Documents/Research/Text2Tabular-Reconstructing-Research-Data-from-Scientific-Publications/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jonasgottal/Documents/Research/Text2Tabular-Reconstructing-Research-Data-from-Scientific-Publications/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


# 10.1161_JAHA.118.011771

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("10.1161_JAHA.118.011771/10.1161_JAHA.118.011771_raw.csv", na_values=[" ", "."], keep_default_na=True,)

# rename columns: site: group, age: Age, bmi:BMI, a1c: HbA1c, hgb: Hemoglobin, wbc: WBC count, plt: Platelet count, plasma_osmol: Plasma osmolality, egfr: eGFR, spot_uacr: Log UACR, sbpbr24h: 24-hour SBP, dia24h: 24-hour DBP, esh_day_dia:  Daytime DBP, esh_day_sbpbr: Daytime SBP, esh_night_sbpbr: Nighttime SBP, esh_night_dia: Nighttime DBP, sex: Gender (Female), current_smoke: Smoker, abpm_htn: Previously diagnosed with hypertension, on_med_htn: Taking antihypertensive medication, abpm_htn: Hypertension, sickle: Sickle cell trait,

rename_dict = {
    "age": "Age",
    "bmi": "BMI",
    "a1c": "HbA1c",
    "hgb": "Hemoglobin",
    "wbc": "WBC count",
    "plt": "Platelet count",
    "plasma_osmol": "Plasma osmolality",
    "egfr": "eGFR",
    "sbpbr24h": "24-hour SBP",
    "dia24h": "24-hour DBP",
    "esh_day_dia": "Daytime DBP",
    "esh_day_sbpbr": "Daytime SBP",
    "esh_night_sbpbr": "Nighttime SBP",
    "esh_night_dia": "Nighttime DBP",
    "sex": "Women",
    "current_smoke": "Smoker",
    "on_med_htn": "Taking antihypertensive medication",
    "sickle": "Sickle cell trait (SCT)",
    "abpm_htn": "Hypertension",
    "thal_any": "a+thalassemia" 
}


df = df.rename(columns=rename_dict)
df["Previously diagnosed with hypertension"] = df["Hypertension"]

df["Log UACR"] = np.log10(df["spot_uacr"])

df["Site"] = df["site"].map({0: "Kilifi", 1: "Nairobi"})
df["SCT_binary"] = df["Sickle cell trait (SCT)"].map({"AA": 0, "AS": 1})
# Sickle cell trait: AS=1, AA=0

df["Sickle cell trait (SCT)"] = df["SCT_binary"]

def create_group(row):
    if row["Site"] == "Kilifi":
        site = "Kilifi"
    elif row["Site"] == "Nairobi":
        site = "Nairobi"
    elif row["Site"] == "Jackson":
        site = "Jackson"
    else:
        site = "Unknown"

    sct_status = "SCT" if row["Sickle cell trait (SCT)"] == 1 else "non-SCT"
    return f"{site} {sct_status}"

df["group"] = df.apply(create_group, axis=1)

# drop agecat3 and age_cat
# Women: female=1, male=0
df["Women"] = df["Women"].map({"female": 1, "male": 0})

# Smoker: Yes=1, No=0  
df["Smoker"] = df["Smoker"].map({"No": 0, "Yes": 1})

# Taking antihypertensive medication: Yes=1, No=0
df["Taking antihypertensive medication"] = df["Taking antihypertensive medication"].map({"No": 0, "Yes": 1})


# a+thalassemia: any deletion=1, normal=0 (if this variable exists)
if "a+thalassemia" in df.columns:
    df["a+thalassemia"] = df["a+thalassemia"].astype("Int64")

# Hypertension should already be binary (0/1)
if "Hypertension" in df.columns:
    df["Hypertension"] = df["Hypertension"].astype(int)

# Previously diagnosed with hypertension - ensure binary
if "Previously diagnosed with hypertension" in df.columns:
    if df["Previously diagnosed with hypertension"].dtype == 'object':
        df["Previously diagnosed with hypertension"] = df["Previously diagnosed with hypertension"].map({"No": 0, "Yes": 1})

# Drop unnecessary columns
columns_to_drop = ["agecat3", "age_cat", "study_no", "site", "SCT_binary", "spot_uacr", "Site", ]
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

df.to_csv("10.1161_JAHA.118.011771/10.1161_JAHA.118.011771.csv", index=False)

variables_to_drop = [
    # Urine chemistry components (raw values)
    "spot_cr",           # Spot urine creatinine (raw values)
    "spot_alb",          # Spot urine albumin (raw values)
    "spot_k",            # Spot urine potassium
    "spot_na",           # Spot urine sodium
    
    # Serum chemistry panel
    "serum_creat",       # Serum creatinine (raw values)
    "serum_k",           # Serum potassium
    "serum_na",          # Serum sodium
    "serum_urea",        # Serum urea
    "cr_mg_per_dl",      # Creatinine in mg/dL format
    
    # Blood pressure pattern classification
    "esh_dip_status",    # BP dipping pattern (dipper/non-dipper/extreme-dipper)
    
    # Combined genetic variables
    "sickle_thal1",      # Combined sickle cell trait and thalassemia variable
    
    # Redundant identifier variables
    "age_cat",           # Age category in text format (redundant)
    
    # Variables already planned to drop
    "agecat3",           # Age category numeric
    "study_no",          # Study identification number
    "site"               # Site identifier (after creating group variable)
]

existing_variables_to_drop = [col for col in variables_to_drop if col in df.columns]
df = df.drop(columns=existing_variables_to_drop)

df.to_csv("10.1161_JAHA.118.011771/10.1161_JAHA.118.011771_clean.csv", index=False)

/Users/jonasgottal/Documents/Research/Text2Tabular-Reconstructing-Research-Data-from-Scientific-Publications/.venv/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


# 10.1186_s13104-019-4632-2

In [15]:
import csv

# Load the dataset
# Load the dataset with proper handling for tab-delimited format
df = pd.read_csv("10.1186_s13104-019-4632-2/10.1186_s13104-019-4632-2.txt", 
                 sep='\t',                    # Tab delimiter
                 #quoting=csv.QUOTE_NONE,      # Don't treat quotes specially during parsing
                 na_values=['', ' ', 'nan'],  # Recognize empty strings as NaN
                 keep_default_na=True,        # Keep default NaN recognition
                 dtype=str)  
# Create rename dictionary to match JSON naming exactly
rename_dict = {
    # Treatment group
    'fluidused': 'group',

    # Demographics
    'cage': 'Age at admission',
    'birthgest_wks': 'Estimated gestational weeks',
    'momage': "Mother's age",

    # Bilirubin measurements
    'tsbadmn': 'TSB at admission',
    'Bilirubinlevelsafterphotother': 'TSB after phototherapy',
    'Changeinbilirubin': 'Change in bilirubin',
    'AN': 'Rate of change in bilirubin',

    # Neurodevelopmental outcomes
    'psychomotorscore': 'Psychomotor score',
    'languagescore': 'Language score',
    'socioemotionalscore': 'Socio-emotional function score',

    # Jaundice causes (these will be used to create binary variables)
    'haemolytic': 'Haemolytic causes',
    'nonhaemolytic': 'Non-haemolytic causes',

    # Outcome
    'outcome': 'Type of discharge',

    # Time-series bilirubin (if needed for calculations)
    'tsb2hrs': 'tsb_2hrs_umol_L',
    'tsb6hrs': 'tsb_6hrs_umol_L', 
    'tsb24hrs': 'tsb_24hrs_umol_L',
    'tsb48hrs': 'tsb_48hrs_umol_L',
    'tsb72hrs': 'tsb_72hrs_umol_L',
    'tsb96hrs': 'tsb_96hrs_umol_L',
    'tsb120hrs': 'tsb_120hrs_umol_L',
    'tsb144hrs': 'tsb_144hrs_umol_L',
    'tsb168hrs': 'tsb_168hrs_umol_L',
}

# Apply the renaming
df = df.rename(columns=rename_dict)

# Convert Y/N to 1/0 for haemolytic cause variables
def convert_yn_to_binary(df):
    """Convert Y/N values to 1/0 for haemolytic variables"""
    yn_columns = ['Haemolytic causes', 'Non-haemolytic causes']
    
    for col in yn_columns:
        if col in df.columns:
            df[col] = df[col].map({'Y': 1, 'N': 0})
    
    return df


# Create binary variables from 'Type of discharge'
def create_binary_from_discharge(df):
    """Create binary variables matching JSON structure"""
    
    # Clean and standardize discharge outcomes first
    if 'Type of discharge' in df.columns:
        df['Type of discharge'] = df['Type of discharge'].replace({
            'Discharged': 'Alive',
            'Exchange_Transfusio': 'Exchange_Transfusion'  # Fix truncated value
        })
    
    # Create binary columns matching JSON structure
    binary_columns = {
        'Repeat phototherapy (Yes)': 'Repeat_phototherapy',
        'Exchange transfusion (Yes)': 'Exchange_Transfusion', 
        'Mortality (Died)': 'Died'
    }
    
    # Initialize all binary columns with 0
    for binary_col in binary_columns.keys():
        df[binary_col] = 0
    
    # Set 1 where conditions match
    for idx, discharge_value in df['Type of discharge'].items():
        if pd.notna(discharge_value):
            for binary_col, discharge_match in binary_columns.items():
                if discharge_value == discharge_match:
                    df.at[idx, binary_col] = 1
    
    return df

# Clean the group values to match JSON exactly
def clean_group_names(df):
    """Clean group names to match JSON naming"""
    df['group'] = df['group'].replace({
        '20_albumin': '20% albumin + phototherapy',
        '10_dextrose': 'Saline + phototherapy'
    })
    return df


# Convert numeric columns
def convert_numeric_columns(df):
    """Convert numeric columns to proper data types"""
    numeric_columns = [
    'Age at admission', "Mother's age", 'Estimated gestational weeks',
    'TSB at admission', 'TSB after phototherapy',
    'Change in bilirubin', 'Rate of change in bilirubin',
    'Psychomotor score', 'Language score', 'Socio-emotional function score',
    'tsb_2hrs_umol_L', 'tsb_6hrs_umol_L', 'tsb_24hrs_umol_L', 'tsb_48hrs_umol_L',
    'tsb_72hrs_umol_L', 'tsb_96hrs_umol_L', 'tsb_120hrs_umol_L',
    'tsb_144hrs_umol_L', 'tsb_168hrs_umol_L'
]
    
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df
# Apply cleaning functions
df = convert_yn_to_binary(df)
df = create_binary_from_discharge(df)
df = clean_group_names(df)
df = convert_numeric_columns(df)
df.to_csv("10.1186_s13104-019-4632-2/10.1186_s13104-019-4632-2.csv", index=False)
df
# Drop unnecessary columns
columns_to_drop = ['tsb_2hrs_umol_L', 'tsb_6hrs_umol_L', 'tsb_24hrs_umol_L',
       'tsb_48hrs_umol_L', 'tsb_72hrs_umol_L', 'tsb_96hrs_umol_L',
       'tsb_120hrs_umol_L', 'tsb_144hrs_umol_L', 'tsb_168hrs_umol_L']
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

df.to_csv("10.1186_s13104-019-4632-2/10.1186_s13104-019-4632-2_clean.csv", index=False)

# 10.1371_journal.pmed.1002015

In [16]:
df = pd.read_csv("10.1371_journal.pmed.1002015/10.1371_journal.pmed.1002015_raw.csv",delimiter=";", na_values=[" ", "."], keep_default_na=True,)

# Complete rename dictionary
rename_dict = {
    "liv_wit_adu_cat": "Number of other persons in house", 
    "chil_liv_wit_cat": "Number of children in household", 
    "live_with_partner": "Live alone",
    "employed": "Employment status",
    "cd4_results_cat": "CD4 count",
    "CrCl_result_cat": "Creatinine clearance",
    "Hb_result_cat": "Hemoglobin",
    "house_primary": "Primary residence",
    "hiv_test_today": "Had HIV test today",
    "hiv_test_prev": "Previous HIV test", 
    "hiv_test_prev_pos": "Previous positive HIV test",
    "attended_prev": "Previously attended clinic",
    "TB_rx": "Currently on TB treatment",
    "TB_sx": "TB symptoms",
    "transport_paid": "Paid for transport",
    "miss_work": "Missed work for clinic",
    "Patient_decision": "Patient decision",
    "Delay_required": "Treatment delay required",

}

df = df.rename(columns=rename_dict)

# Employment status mapping
employment_map = {
    "Employed formally": "Employed formally",
    "Work informally": "Work informally", 
    "Unemployed and seeking work": "Unemployed, seeking work",
    "Unemployed and not seeking work": "Unemployed, not seeking work"
}
df["Employment status"] = df["Employment status"].map(employment_map)

# CD4 count mapping (using midpoint values from README categories)
cd4_map = {
    1: 180,   # <200, using 180 as representative
    2: 350,   # ≥200-≤499, using 350 as midpoint  
    3: 600    # ≥500, using 600 as representative
}
df["CD4 count"] = df["CD4 count"].map(cd4_map)

# Creatinine clearance mapping (using midpoint values)
crcl_map = {
    1: 25,    # ≤29.0
    2: 45,    # ≥30.0-≤59.0  
    3: 75,    # ≥60.0-≤89.0
    4: 100    # ≥90.0
}
df["Creatinine clearance"] = df["Creatinine clearance"].map(crcl_map)

# Hemoglobin mapping (using midpoint values)
hb_map = {
    1: 6.5,   # ≤7.0
    2: 8.5,   # ≥7.1-≤9.9
    3: 11.0   # ≥10.0
}
df["Hemoglobin"] = df["Hemoglobin"].map(hb_map)

# Household composition mapping
household_adults_map = {
    1: 2,     # min-3, using 2 as representative
    2: 5,     # 4-6, using 5 as midpoint
    3: 8      # 7-max, using 8 as representative  
}
df["Number of other persons in house"] = df["Number of other persons in house"].map(household_adults_map)

household_children_map = {
    1: 2,     # min-3
    2: 5,     # 4-6  
    3: 8      # 7-max
}
df["Number of children in household"] = df["Number of children in household"].map(household_children_map)

# Primary residence mapping
residence_map = {
    "This is my primary house": "Yes",
    "My primary house is in another location": "No"
}
df["Primary residence"] = df["Primary residence"].map(residence_map)

# Group assignment mapping
df["group"] = df["group"].map({
    "RAPID (randomized to rapid initiation)": "Rapid arm", 
    "STANDARD (randomized to standard initiation)": "Standard arm"
})

# Patient decision mapping
decision_map = {
    "Initiate today": "Initiate today",
    "Return within 4 weeks": "Return within 4 weeks", 
    "Return later": "Return later",
    "Other": "Other"
}
df["Patient decision"] = df["Patient decision"].map(decision_map)

# Live alone mapping (swap yes/no since it's asking opposite question)
df["Live alone"] = df["Live alone"].map({"Yes": "No", "No": "Yes"})

# Convert yes/no columns to 1/0
yes_no_cols = []
for col in df.columns:
    if col in df.columns:
        values = df[col].dropna().astype(str).str.lower().unique()
        if set(values) <= {"yes", "no"}:
            yes_no_cols.append(col)

for col in yes_no_cols:
    df[col] = df[col].astype(str).str.lower().map({"yes": 1, "no": 0}).astype("Int64")

# Handle specific categorical columns that should remain as strings
categorical_cols = ["Employment status", "Patient decision", "group", "Primary residence", "Live alone"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace('nan', pd.NA)

df.to_csv("10.1371_journal.pmed.1002015/10.1371_journal.pmed.1002015.csv", index=False)

# Drop unnecessary columns
columns_to_drop = ['id', 'eligible', 'referral', 'referred_by',  'disclosed_home', 'other_positive_home', 'other_art_home', 'no_others_on_art', 'yes_spouse_on_art', 'yes_child_on_art', 'yes_family_on_art', 'yes_friend_on_art', 'yes_other_on_art','activity', 'grant', 'transport_mode','transport_cost', 'help_paid', 'help_cost','miss_work_cost', 'Delay_reason_late', 'Delay_reason_tb', 'Delay_reason_cond', 'Delay_reason_ready', 'Delay_reason_other', 'Delay_reason_other_notes', 'TB_rx_14d', 'TB symptoms', 'Xpert_sample_taken', 'Xpert_result_1',       'Xpert_result_', 'TB_clinical_required', 'TB_clinical_result', 'TB_notes', 'Stage3_4_cond', 'Cond_delay', 'Stage3_4_cond_notes',       'Confirm_eligible', 'Subject_withdraw', 'Subject_refuse', 'Preg_test_result', 'Exam_result', 'Edu_session_done',       'Nurse_review',]
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)
df.to_csv("10.1371_journal.pmed.1002015/10.1371_journal.pmed.1002015_clean.csv", index=False)

# 10.1371_journal.pmed.1002785


In [17]:
import pandas as pd
import numpy as np
import json

# Step 2: Load LEAN dataset
df = pd.read_excel('10.1371_journal.pmed.1002785/10.1371_journal.pmed.1002785.xls')
# Step 3: Create exact mapping from LEAN variables to JSON format

exact_variable_mapping = {
    'study_id': 'study_id',
    'Interven.enrollmentmarch_arm_1': 'group',

    'age_n.baseline': 'Age (years)',
    'education.baseline': 'Education (years)',
    'income_care_n.baseline': 'Patient income last month (RMB)',
    'income_family_total_n.baseline': 'Family annual income (RMB)',
    'duration_of_illness_n.baseline': 'Duration of schizophrenia (years)',
    # Caregiver age (years) - if available, add real column from dataset

    'Antipsychotic_drug_adherence': 'Pill-count adherence',
    'DAI_n.baseline': 'DAI-10',
    'BAR_n.baseline': 'BARS',
    'pharmacy_record_n.endline': 'Refill record',
    'WHODAS_n.endline': 'WHODAS',
    'GASS_n.baseline': 'GASS',

    'cgi1_n.baseline': 'CGI–Severity',
    'cgi2_n.baseline': 'CGI–Improvement',

    'sex_n.baseline': 'Patient gender (Female)',
    'marriage_n.baseline': 'Patient married',
    'occupation_n.baseline': 'Patient employed',
    'living_status_care_n.baseline': 'Patient living alone',

    'education.baseline': 'Literate',  # Will derive from education years
    # Add caregiver fields if dataset provides them

    'NrHosp': 'Re-hospitalization due to schizophrenia',
    'Dead': 'Death for any reason',
    'substance.abuse.comorbidity': 'Substance abuse',
    'NrWonder': 'Wandering',
    'NrViolence': 'Violence against others',
    'NrDamage': 'Damaging goods',

    # Relapse, Suicide, Self-harm, Caregiver variables: map if columns exist
    # Top antipsychotic prescribed handled separately
}

# Step 4: Apply the exact mapping
df_transformed = df.rename(columns=exact_variable_mapping)

# Step 5: Create the exact group labels to match JSON
df_transformed['group'] = df_transformed['group'].map({
    1: 'Intervention',
    0: 'Control'
})

# Step 6: Transform categorical variables to match JSON format exactly
if 'Patient gender (Female)' in df_transformed.columns:
    df_transformed['Patient gender (Female)'] = (df_transformed['Patient gender (Female)'] == 2).astype(int)
if 'Patient married' in df_transformed.columns:
    df_transformed['Patient married'] = (df_transformed['Patient married'] == 1).astype(int)
if 'Patient employed' in df_transformed.columns:
    df_transformed['Patient employed'] = (df_transformed['Patient employed'] > 0).astype(int)
if 'Patient living alone' in df_transformed.columns:
    df_transformed['Patient living alone'] = (df_transformed['Patient living alone'] == 1).astype(int)
if 'Literate' in df_transformed.columns:
    df_transformed['Literate'] = (df_transformed['Literate'] >= 3).astype(int)

    

# Handle missing values (999 typically indicates missing)
missing_value_cols = [
    'Clinical Global Impression (CGI)\u2013Severity', 'Clinical Global Impression (CGI)\u2013Improvement',
    'WHODAS score', 'DAI-10 adherence', 'BARS adherence', 'CGI\u2013Severity score',
    'CGI\u2013Improvement score', 'Glasgow Antipsychotic Side-effect Scale (GASS) score'
]

for col in missing_value_cols:
    if col in df_transformed.columns:
        df_transformed[col] = df_transformed[col].replace(999, np.nan)

# Create binary variables for events (convert counts to binary)
event_vars = ['Re-hospitalization due to schizophrenia', 'Wandering', 'Violence against others', 'Damaging goods']
for var in event_vars:
    if var in df_transformed.columns:
        df_transformed[var] = (df_transformed[var] > 0).astype(int)


df_transformed.to_csv("10.1371_journal.pmed.1002785/10.1371_journal.pmed.1002785.csv", index=False)

columns_to_drop = [
    # Technology usage variables (baseline) - not documented in PDF
    'sms_usage_n.baseline',
    'sms_usage_lay_n.baseline', 
    'phone_ownership_n.baseline',
    'phone_ownership_lay_n.baseline',
    
    # Medication adherence measures not mentioned in PDF
    'Morisky_n.baseline',
    'Morisky_n.endline',
    'medication_supervised_n.baseline',
    
    # Additional adherence outcomes not documented
    'All_drug_adherence',
    'Prescription_consistency',
    
    # Demographic detail (PDF only reports age, not birth year)
    'year_of_birth_n.baseline'
]

existing_columns_to_drop = [col for col in columns_to_drop if col in df_transformed.columns]
df_transformed = df_transformed.drop(columns=existing_columns_to_drop)
df_transformed.to_csv("10.1371_journal.pmed.1002785/10.1371_journal.pmed.1002785_clean.csv", index=False)


# 10.1371_journal.pmed.1003621

In [18]:
df = pd.read_csv("10.1371_journal.pmed.1003621/10.1371_journal.pmed.1003621_raw.csv")

def safe_convert_to_binary(series):
        """Safely convert a series to binary (0/1) handling various input types"""
        # First convert to string to handle any data type
        series_str = series.astype(str)
        # Replace various representations of missing/empty values
        series_clean = series_str.replace(['nan', 'NaN', 'None', '', ' ', 'null'], np.nan)
        # Convert to numeric, coercing errors to NaN
        series_numeric = pd.to_numeric(series_clean, errors='coerce')
        # Fill NaN with 0 and convert to int
        return series_numeric.fillna(0).astype(int)

# Updated rename dictionary to match JSON structure exactly
rename_dict = {
        # Group assignment
        "INT": "group",
        
        # Demographics - match JSON naming exactly
        "BL_age_cat": "Age",
        "gender_cluster": "Gender (Female)",
        "BL_education": "Education",
        "BL_occupation": "Occupation",
        "BL_caste_cat": "Caste category",
        "BL_religion": "Religion",
        "BL_marital_status": "Marital status",
        "BL_lang_used": "Primary language",
        "BL_live_with": "Who do you live with?",
        
        # Health conditions
        "BL_chronic_disease": "Chronic disease",
        "BL_cancer": "Cancer",
        "BL_diabetes": "Diabetes",
        "BL_hypertension": "Hypertension",
        "BL_asthma": "Asthma",
        "BL_chronic_other": "Other chronic disease",
        
        # Mental health service history
        "medication": "Ever taken medication for mental health problems",
        "counselling": "Ever received counseling services (number of times)",

        # Mental health scales - continuous
        "GHQ_total": "GHQ-12 score",
        "PHQ9_total": "PHQ-9 score",
        "PCL_total": "PCL score",
        "MSPSS_total": "MSPSS score",
        "SSS_total": "SSS-8 score",
        "WHODAS_total": "WHODAS score",
        "RTC_total": "RTC score",
        
        # Clinical outcomes
        "PHQ9_red_yn": "50% reduction in PHQ-9 from baseline",
        "heart_mind_ny": "Heart–mind problems",  # note: en-dash
        
        # Traumatic events
        "BL_te_1_nd": "Ever experienced a natural disaster",
        "BL_te_2_accid": "Been in a serious accident",
        "BL_te_3_sick": "Had a serious sickness",
        "BL_te_4_war": "Been in the military or war zone",
        "BL_te_5_death": "Seen/had a death/murder of close family or friend",
        "BL_te_6_suic": "Seen/had close friend/family member commit suicide",
        "BL_te_7_gun_knife": "Been attacked with a gun/knife",
        "BL_te_8_attack": "Been attacked without weapon",
        "BL_te_9_beat_child": "Beaten as a child",
        "BL_te_10_adult_sex_before13": "Had adult sexual contact before age 13",
        "BL_te_11_sex_after13": "Had unwanted sexual contact after age 13",
        
        # Economic indicators
        "BL_ses_concrete": "Concrete building",
        "BL_ses_elec": "Electricity",
        "BL_ses_drink_water": "Drinking water",
        "BL_ses_radio": "Radio",
        "BL_ses_tv": "Television",
        "BL_ses_mobile": "Simple mobile phone",
        "BL_ses_mobile_smart": "Smart mobile phone",  # corrected typo
        "BL_ses_cycle": "Bicycle",
        "BL_ses_lp_gas": "LP gas"
    }
    
# Apply renaming
df = df.rename(columns=rename_dict)


df["group"] = df["group"].map({
    0: "Control",
    1: "Group PM+"
})

# Gender mapping - create both categorical and binary versions
df["Gender (Female)"] = (df["Gender (Female)"] == 1).astype(int)

counseling_map = {
    0: "0 time",
    1: "1 to 4 times",
    2: "5 to 10 times",
    3: ">10 times"
}
df["Ever received counseling services (number of times)"] = df["Ever received counseling services (number of times)"].map(counseling_map)

# Age - keep as continuous (convert categories to midpoints)
age_map = {1: 25, 2: 35, 3: 45, 4: 55, 5: 65, 6: 75}
df["Age"] = df["Age"].map(age_map)

education_map = {
    1: "Cannot read or write",
    2: "Literate or informal education",
    3: "Primary level",
    4: "Secondary",
    5: "Higher secondary",
    6: "University"
}
df["Education"] = df["Education"].map(education_map)

# Occupation mapping
occupation_map = {
    1: "Farmer",
    2: "Business or job",
    3: "Daily wage laborer",
    4: "Unemployed",
    5: "Student",
    6: "Housewife",
    7: "Other"
}
df["Occupation"] = df["Occupation"].map(occupation_map)

# Caste category mapping
caste_map = {
    1: "Upper caste (Brahman, Chhetri)",
    2: "Janajati",
    3: "Madhesi and Local Indigenous",
    4: "Other"
}
df["Caste category"] = df["Caste category"].map(caste_map)


# Religion mapping
religion_map = {
    1: "Hindu",
    2: "Buddhist",
    3: "Muslim",
    4: "Christian",
    5: "No religion",
    6: "Other"
}
df["Religion"] = df["Religion"].map(religion_map)


# Marital status mapping
marital_map = {
    1: "Unmarried",
    2: "Married",
    3: "Widowed",
    4: "Divorced",
    5: "Separated"
}
df["Marital status"] = df["Marital status"].map(marital_map)

# Primary language mapping
language_map = {
    1: "Nepali",
    2: "Other"
}
df["Primary language"] = df["Primary language"].map(language_map)


# Who do you live with mapping
living_map = {
    1: "Living alone",
    2: "With 1 other person",
    3: "With 2 to 3 other people",
    4: "With 4 or more other people"
}
df["Who do you live with?"] = df["Who do you live with?"].map(living_map)


# Ordinal variable: Ever received counseling services
counseling_map = {
    0: "0 time",
    1: "1 to 4 times",
    2: "5 to 10 times",
    3: ">10 times"
}
df["Ever received counseling services (number of times)"] = df["Ever received counseling services (number of times)"].map(counseling_map)





# Economic indicators - already renamed above, just ensure binary
economic_vars = [
    "Concrete building present", "Electricity present", "Drinking water present",
    "Radio present", "Television present", "Simple mobile phone present", 
    "Smart mobile phone present", "Bicycle present", "LP gas present"
]

for var in economic_vars:
        if var in df.columns:
            df[var] = safe_convert_to_binary(df[var])

# Traumatic events - already renamed, ensure binary
trauma_vars = [
    "Ever experienced a natural disaster", "Been in a serious accident",
    "Had a serious sickness", "Been in the military or war zone",
    "Seen/had a death/murder of close family or friend",
    "Seen/had close friend/family member commit suicide",
    "Been attacked with a gun/knife", "Been attacked without weapon",
    "Beaten as a child", "Had unwanted sexual contact after age 13"
]

for var in trauma_vars:
    if var in df.columns:
        df[var] = safe_convert_to_binary(df[var])

# Mental health service history
df["Ever taken medication for mental health problems"] = safe_convert_to_binary(df["Ever taken medication for mental health problems"])


# Clinical outcomes

df["50% reduction in PHQ-9 from baseline"] = safe_convert_to_binary(df["50% reduction in PHQ-9 from baseline"])


binary_vars = ["Chronic disease", "Cancer", "Diabetes", "Hypertension", "Asthma", "Other chronic disease",
               "Concrete building", "Electricity", "Drinking water", "Radio", "Television",
               "Simple mobile phone", "Smart mobile phone", "Bicycle", "LP gas",
               "Ever taken medication for mental health problems", "Ever experienced a natural disaster",
               "Been in a serious accident", "Had a serious sickness", "Been in the military or war zone",
               "Seen/had a death/murder of close family or friend", "Seen/had close friend/family member commit suicide",
               "Been attacked with a gun/knife", "Been attacked without weapon", "Beaten as a child",
               "Had adult sexual contact before age 13", "Had unwanted sexual contact after age 13",
               "Heart–mind problems", "50% reduction in PHQ-9 from baseline"]
for var in binary_vars:
    if var in df.columns:
        df[var] = safe_convert_to_binary(df[var])

        

# save
df.to_csv("10.1371_journal.pmed.1003621/10.1371_journal.pmed.1003621.csv", index=False)


# cols to drop: participant_id, cluster, mh_access, urban, disaster_risk, visit, BL_te_1_nd_when, sumses, completeses
cols_to_drop = [
    "participant_id",
    "cluster",
    "mh_access",
    "urban",
    "disaster_risk",
    "visit",
    "BL_te_1_nd_when",
    "sumses",
    "completeses"]

existing_columns_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)
# save
df.to_csv("10.1371_journal.pmed.1003621/10.1371_journal.pmed.1003621_clean.csv", index=False)

# 10.30802_AALAS-JAALAS-23-000028

In [3]:
file_path = "10.30802_AALAS-JAALAS-23-000028/10.30802_AALAS-JAALAS-23-000028.xlsx"
cd1_igs = pd.read_excel(file_path, sheet_name='CD-1 IGS')
balbcj = pd.read_excel(file_path, sheet_name='BALBcJ')
cd1_igs['STRAIN'] = 'CD-1' 
balbcj['STRAIN'] = 'BALB/cJ'
balbcj['TRT'] = balbcj['TRT'].map({1: 2, 2: 1})
combined_data = pd.concat([cd1_igs, balbcj], ignore_index=True)
trt_map = {1: 'Tail-lift', 2: 'Tunnel'}
combined_data['TRT'] = combined_data['TRT'].map(trt_map)
# combine strain and treatment into a single column
combined_data['group'] = combined_data['STRAIN'] + " " + combined_data['TRT']
# drop TRT col
combined_data = combined_data.drop(columns=['STRAIN', 'TRT'])
name_mapping = {
    # Ordinal variables
    'PUPS': 'Pups born per litter',
    'WEAN': 'Pups weaned per litter',

    # Continuous variables  
    'LITT_WT': 'Total litter weight at weaning',
    'PUP_WT': 'Pup weight at weaning',
    'LITT_INT': 'Interlitter interval',
    'PI': 'Productivity index',

    # Binary variables
    'LITTER': 'Number of litters',

    # Categorical variables (sex specific)
    'M': 'Male',
    'F': 'Female',

    # Administrative variables (can be dropped later)
    'ID': 'cage_id',
    'POS': 'cage_position_alphanumeric', 
    'COL': 'rack_column_vertical',
    'ROW': 'rack_row_horizontal',
    'DOM': 'date_of_mating',
    'DOB': 'date_of_birth', 
    'DOW': 'date_of_weaning',
}

# Apply the exact JSON variable name mapping
combined_data = combined_data.rename(columns=name_mapping)
combined_data['Sex ratio F/(M + F)'] = (
    combined_data['Female'] /
    (combined_data['Female'] + combined_data['Male'])
)


combined_data.to_csv("10.30802_AALAS-JAALAS-23-000028/10.30802_AALAS-JAALAS-23-000028.csv", index=False)

cols_to_drop = [
    'cage_id',
    'cage_position_alphanumeric', 
    'rack_column_vertical',
    'rack_row_horizontal',
    'date_of_mating',
    'date_of_birth',
    'date_of_weaning'
]

existing_columns_to_drop = [col for col in cols_to_drop if col in combined_data.columns]
combined_data = combined_data.drop(columns=existing_columns_to_drop)

combined_data.to_csv("10.30802_AALAS-JAALAS-23-000028/10.30802_AALAS-JAALAS-23-000028_clean.csv", index=False)
